# 4. Linear Regression

Linear regression as a model is interpretable. Get a number for each feature that directly represents how that feature influences the target (dependent variable). If the coefficient is larger, it potentially means that the variable has a larger impact both directly (positive) or inversely (negative) to a large degree.

Linear regression first estimates a baseline value called the intercept, then estimates a multiplicative coefficient for each feature. Summing the baseline and all the coefficient-transformed features, to get the final prediction.

Makes it __accessible to general interpretation__ by inspection of a weights vector

# Linear regression with TensorFlow

TensorFlow achieves the result in iteratively, gradually learning the best linear regression parameters that will minimize the loss.

`tf.estimators` is an API that has pre-made specific procedures to simplify training, evaluation, predicting and the exporting of models (e.g regression models or basic neural networks). Easily deployed on any hardware or on distributed multi-server environments without much code changes

## Development with Estimators

4 steps involved in developing an Estimator model:
1. Acquire data using `tf.data` functions
2. Instantiate the feature columns
3. Instantiate and train the Estimator
4. Evaluate the model's performance

Below will loop through batches of data points and let TensorFlow update the slope and y intercept. Using the _Boston Housing_ dataset. Create a regression Estimator in TensorFlow using Keras functions

In [16]:
import tensorflow as tf
import numpy as np
import pandas as pd
import tensorflow_datasets as tfds
tfds.disable_progress_bar()

# Get the Boston Housing dataset
housing_url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data'
path = tf.keras.utils.get_file(housing_url.split('/')[-1], housing_url)
columns = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
data = pd.read_table(
	path,
	delim_whitespace=True,
	header=None,
	names=columns
)
data.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [2]:
data.describe()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
count,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000
mean,3.613524,11.363636,11.136779,0.069170,0.554695,6.284634,68.574901,3.795043,9.549407,408.237154,18.455534,356.674032,12.653063,22.532806
std,8.601545,23.322453,6.860353,0.253994,0.115878,0.702617,28.148861,2.105710,8.707259,168.537116,2.164946,91.294864,7.141062,9.197104
min,0.006320,0.000000,0.460000,0.000000,0.385000,3.561000,2.900000,1.129600,1.000000,187.000000,12.600000,0.320000,1.730000,5.000000
25%,0.082045,0.000000,5.190000,0.000000,0.449000,5.885500,45.025000,2.100175,4.000000,279.000000,17.400000,375.377500,6.950000,17.025000
50%,0.256510,0.000000,9.690000,0.000000,0.538000,6.208500,77.500000,3.207450,5.000000,330.000000,19.050000,391.440000,11.360000,21.200000
75%,3.677082,12.500000,18.100000,0.000000,0.624000,6.623500,94.075000,5.188425,24.000000,666.000000,20.200000,396.225000,16.955000,25.000000
max,88.976200,100.000000,27.740000,1.000000,0.871000,8.780000,100.000000,12.126500,24.000000,711.000000,22.000000,396.900000,37.970000,50.000000


In [17]:
# Splitting Training and Testing Data
np.random.seed(1)
train = data.sample(frac=0.8).copy()
y_train = train['MEDV']
train.drop('MEDV', axis=1, inplace=True)
test = data.loc[~data.index.isin(train.index)].copy()
y_test = test['MEDV']
test.drop('MEDV', axis=1, inplace=True)

In [14]:
learning_rate = 0.05
def make_input_fn(data_df, label_df, num_epochs=10, shuffle=True, batch_size=256):
	''' function creates a tf.data dataset from a pandas DataFrame turning into a Python
		dictionary of pandas Series. Also provides batch size definitions and random shuffling
	'''
	def input_function():
		ds = tf.data.Dataset.from_tensor_slices((dict(data_df), label_df))
		if shuffle:
			ds = ds.shuffle(1000)
		ds = ds.batch(batch_size).repeat(num_epochs)
		print(ds)
		return ds

	return input_function

def define_feature_column(data_df, categorical_cols, numeric_cols):
    '''function that maps each column name to a specific tf.feature_column transformation
       offering functions that can process any kind of data
       
       numeric: numeric_column
       categorical: cateorical_column_with_vocabulary_list
    '''
    feature_columns = []
    for feature_name in numeric_cols:
        feature_columns.append(tf.feature_column.numeric_column(
			feature_name,
			dtype=tf.float32
		))
    
    for feature_name in categorical_cols:
        vocabulary = data_df[feature_name].unique()
        feature_columns.append(tf.feature_column.categorical_column_with_vocabulary_list(
			feature_name,
			vocabulary
		))

    return feature_columns

def canned_keras(model, output_dir=None):
    if output_dir is None:
        model_dir = tempfile.mkdtemp()
    else:
        model_dir = output_dir
    keras_estimator = tf.keras.estimator.model_to_estimator(
		keras_model=model,
		model_dir=model_dir, 
		config=tf.estimator.RunConfig().replace(save_summary_steps=100)
	)
    return keras_estimator

In [21]:
# Declare the different data types for each column
categorical_cols = ['CHAS', 'RAD']
numeric_cols = ['CRIM', 'ZN', 'INDUS',  'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'B', 'LSTAT']

# Define the feature columns using the above functions
feature_columns = define_feature_column(data, categorical_cols, numeric_cols)
train_input_fn = make_input_fn(train, y_train, num_epochs=1400)
test_input_fn = make_input_fn(test, y_test, num_epochs=1, shuffle=False)

# Define the model
output_dir = './logs/LinearRegressor_1'
linear_est = tf.estimator.LinearRegressor(
	feature_columns=feature_columns,
	model_dir=output_dir,
	config=tf.estimator.RunConfig().replace(save_summary_steps=100)
)

# Train the model
linear_est.train(train_input_fn)
result = linear_est.evaluate(test_input_fn)
print(result)

INFO:tensorflow:Using config: {'_model_dir': './logs/LinearRegressor_1', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
<RepeatDataset shapes: ({CRIM: (None,), ZN: (None,), INDUS: (None,), CHAS: (None,), NOX: (None,), RM: (None,), AGE: (None,), DIS: (None,), RAD: (No

Estimator sifts the data from the data functions and converts the data into proper form based on the matched feature name and `tf.feature_column` function does the entire job.

Optimal line found by Estimator might not be the line of best fit. Depends on the number of iterations, learning rate, loss function and other hyperparameters of the algorithm. Observe the loss function over time to help troubleshoot problems

### Extensions
 
__Combining 2 variables__

A combination between 2 variables can explain the target better than the features taken singularly. The combination could capture an underlying component of the prediction that individually the variables cannot explain. Using `tf.feature_column_.crossed_column` function can automatically allow Estimator to receive the output among the features.

In [11]:
def create_interactions(interactions_list, buckets=5):
	interactions = []
	for (a, b) in interactions_list:
		interactions.append(tf.feature_column.crossed_column([a, b], hash_bucket_size=buckets))
	return interactions

# Declare the different data types for each column
categorical_cols = ['CHAS', 'RAD']
numeric_cols = ['CRIM', 'ZN', 'INDUS',  'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'B', 'LSTAT']

# Define the feature columns using the above functions
feature_columns = define_feature_column(data, categorical_cols, numeric_cols)
train_input_fn = make_input_fn(train, y_train, num_epochs=1400)
test_input_fn = make_input_fn(test, y_test, num_epochs=1, shuffle=False)
derived_feature_columns = create_interactions([['RM', 'LSTAT']])

# Define the model
output_dir = './logs/LinearRegressor_1'
linear_est = tf.estimator.LinearRegressor(
	feature_columns=feature_columns+derived_feature_columns,
	model_dir = output_dir,
	config=tf.estimator.RunConfig().replace(save_summary_steps=100)
)
linear_est.train(train_input_fn)
result = linear_est.evaluate(test_input_fn)
print(result)

INFO:tensorflow:Using config: {'_model_dir': './logs/LinearRegressor_1', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
Instructions for updating:
If using Keras pass *_constraint arguments to layers.
Instructions for updating:
Use Variable.read_value. Variables in 2

In [16]:
%load_ext tensorboard
%tensorboard --logdir ./logs/LinearRegressor_1

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 15656), started 0:00:55 ago. (Use '!kill 15656' to kill it.)

Fitting is much faster and more stable compared to the previous model => provided more informative features to the model

In [17]:
# For handling predictions
def dicts_to_preds(pred_dicts):
    return np.array([pred['predictions'] for pred in pred_dicts])
preds = dicts_to_preds(linear_est.predict(test_input_fn))
print(preds)

<RepeatDataset shapes: ({CRIM: (None,), ZN: (None,), INDUS: (None,), CHAS: (None,), NOX: (None,), RM: (None,), AGE: (None,), DIS: (None,), RAD: (None,), TAX: (None,), PTRATIO: (None,), B: (None,), LSTAT: (None,)}, (None,)), types: ({CRIM: tf.float64, ZN: tf.float64, INDUS: tf.float64, CHAS: tf.int64, NOX: tf.float64, RM: tf.float64, AGE: tf.float64, DIS: tf.float64, RAD: tf.int64, TAX: tf.float64, PTRATIO: tf.float64, B: tf.float64, LSTAT: tf.float64}, tf.float64)>
INFO:tensorflow:Calling model_fn.

If you intended to run this layer in float32, you can safely ignore this warning. If in doubt, this warning is likely only an issue if you are porting a TensorFlow 1.X model to TensorFlow 2.

To change all layers to have dtype float64 by default, call `tf.keras.backend.set_floatx('float64')`. To change just this layer, pass dtype='float64' to the layer constructor. If you are the author of this layer, you can disable autocasting by passing autocast=False to the base Layer constructor.

INFO

# Turning a Keras model into an Estimator

`tf.estimator` module is good for out-of-the-box models that have flexible deployment options and are scalable, but lack the flexibility in model architecture. Keras instead offers the modularity, which can be combined with Estimators to take advantage of both APIs

Achieve deployability with modularity of models. Reuse the Boston Housing dataset

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import tensorflow_datasets as tfds
tfds.disable_progress_bar()
import tensorflow.keras as keras

In [18]:
# Redefine feature column function to specify input to Keras model
def define_feature_columns_layers(data_df, categorical_cols, numerical_cols):
    feature_columns = []
    feature_layer_inputs = {}
    
    for feature_name in numerical_cols:
        feature_columns.append(tf.feature_column.numeric_column(feature_name, dtype=tf.float32))
        feature_layer_inputs[feature_name] = tf.keras.Input(shape=(1, ), name=feature_name)
    
    for feature_name in categorical_cols:
        vocabulary = data_df[feature_name].unique()
        cat = tf.feature_column.categorical_column_with_vocabulary_list(feature_name, vocabulary)
        cat_one_hot = tf.feature_column.indicator_column(cat)
        feature_columns.append(cat_one_hot)
        feature_layer_inputs[feature_name] = tf.keras.Input(shape=(1, ), name=feature_name, dtype=tf.int64)
        # note that the shape for keras.Input is the column, row is automatically None
    
    return feature_columns, feature_layer_inputs 

# Redefine the interaction (combination of 2 variables) feature as well
def create_interactions(interactions_list, buckets=5):
    feature_columns = []

    for (a, b) in interactions_list:
        crossed_features = tf.feature_column.crossed_column([a, b], hash_bucket_size=buckets)
        crossed_features_one_hot = tf.feature_column.indicator_column(crossed_features)
        feature_columns.append(crossed_features_one_hot)
    
    return feature_columns
	
# Define model
def create_linreg(feature_columns, feature_layer_inputs, optimizer):
    feature_layer = keras.layers.DenseFeatures(feature_columns)
    feature_layer_outputs = feature_layer(feature_layer_inputs)
    norm = keras.layers.BatchNormalization()(feature_layer_outputs)
    outputs = keras.layers.Dense(1, kernel_initializer='normal', activation='linear')(norm)
    
    model = keras.Model(inputs=[v for v in feature_layer_inputs.values()], outputs=outputs)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    
    return model

In [8]:
# Create the model 
categorical_cols = ['CHAS', 'RAD']
numeric_cols = ['CRIM', 'ZN', 'INDUS',  'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'B', 'LSTAT']
feature_columns, feature_layer_inputs = define_feature_columns_layers(data, categorical_cols, numeric_cols)
interaction_columns = create_interactions([['RM', 'LSTAT']])
feature_columns += interaction_columns
optimizer = keras.optimizers.Ftrl(learning_rate=0.02)
model = create_linreg(feature_columns, feature_layer_inputs, optimizer)

In [31]:
# Converting the model into an estimator
import tempfile
def canned_keras(model, output_dir=None):
    if output_dir is None:
        model_dir = tempfile.mkdtemp()
    else:
        model_dir = output_dir
    keras_estimator = tf.keras.estimator.model_to_estimator(
		keras_model=model,
		model_dir=model_dir, 
		config=tf.estimator.RunConfig().replace(save_summary_steps=100)
	)
    return keras_estimator
estimator = canned_keras(model)

# Train the model and evaluate
train_input_fn = make_input_fn(train, y_train, num_epoch=1400)
test_input_fn = make_input_fn(test, y_test, num_epoch=1, shuffle=False)
estimator.train(train_input_fn)
result = estimator.evaluate(test_input_fn)
print(result)

INFO:tensorflow:Using the Keras model provided.
INFO:tensorflow:Using config: {'_model_dir': 'C:\\Users\\MARTIN~1\\AppData\\Local\\Temp\\tmp7vbiwrxj', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}


TypeError: make_input_fn() got an unexpected keyword argument 'num_epoch'

Canned Keras Esimators are a quick robust way to combine the flexibility of user-defined models in keras with the high-performance training and deployment of estimators.

# 4c. Understanding Loss Functions in Linear Regression

Goal: Understand the effect of loss function in algorithm convergence.

Illustrate how the L1 and L2 loss functions affect convergence and predictions in linear regression. Continue to use the Boston Housing dataset.

In [32]:
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import pandas as pd
import tensorflow_datasets as tfds
tfds.disable_progress_bar()

In [33]:
# Redefine create_linreg by adding new parameter to control the loss function
def create_linreg(feature_columns, feature_layer_inputs, optimizer, loss='mean_squared_error', metrics=['mean_absolute_error']):
    feature_layer = keras.layers.DenseFeatures(feature_columns)
    feature_layer_outputs = feature_layer(feature_layer_inputs)
    norm = keras.layers.BatchNormalization()(feature_layer_outputs)
    outputs = keras.layers.Dense(1, kernel_initializer='normal', activation='linear')(norm)
    
    model = keras.Model(inputs=[v for v in feature_layer_inputs.values()], outputs=outputs)
    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
    return model

In [34]:
# Defining the columns
categorical_cols = ['CHAS', 'RAD']
numeric_cols = ['CRIM', 'ZN', 'INDUS',  'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'B', 'LSTAT']
feature_columns, feature_layer_inputs = define_feature_columns_layers(data, categorical_cols, numeric_cols)
interactions_columns = create_interactions([['RM', 'LSTAT']])

feature_columns += interactions_columns

optimizer = keras.optimizers.Ftrl(learning_rate=0.02)

# Create the model with L1 loss instead
model = create_linreg(
    feature_columns, 
    feature_layer_inputs, 
    optimizer, 
    loss='mean_absolute_error', 
    metrics=['mean_absolute_error', 'mean_squared_error']
)
estimator = canned_keras(model, output_dir='logs/L1_Loss')
train_input_fn = make_input_fn(train, y_train, num_epochs=1400)
test_input_fn = make_input_fn(test, y_test, num_epochs=1, shuffle=False)
estimator.train(train_input_fn)
result = estimator.evaluate(test_input_fn)
print(result)

INFO:tensorflow:Using the Keras model provided.
INFO:tensorflow:Using config: {'_model_dir': 'logs/L1_Loss', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
Instructions for updating:
Use Variable.read_value. Variables in 2.X are initialized automatically both in eage

In [38]:
%load_ext tensorboard
%tensorboard --logdir ./logs/L1_Loss

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## On Learning Rates

Too small a learning rate and the convergence will be slow, too large a learning rate and the algorithm might overshoot the minima. Furthermore, different algorithms have different "reactions" to different learning rates. 

For e.g L2 norms are more sensitive to large learning rates due to the design of the loss function as compared to L1 norms

Therefore it's important to select the learning rate based on algorithm, scenario and step

# 4d. Implementing Lasso and Ridge Regression

Regularization methods are used to limit the influence of the coefficients on the regression output. The regularization term is added to the back of the loss function to limit the slopes in the algorithm. 

This limits the number/impact of features have on the impact of dependent features

$$
L_{lasso}(\hat{\beta}) = \sum_{i=1}^{n}(y_i-x'_i\hat{\beta})^2+\lambda \sum_{j=1}^{m}|\hat{\beta}_j| \\
L_{ridge}(\hat{\beta}) = \sum_{i=1}^{n}(y_i-x'_i\hat{\beta})^2+\lambda \sum_{j=1}^{m}w_j\hat{\beta}_j^2 \\
$$

In [59]:
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import pandas as pd
import tensorflow_datasets as tfds
tfds.disable_progress_bar()

def create_ridge_linreg(feature_columns, feature_layer_inputs, optimizer, loss='mean_squared_error', metrics=['mean_absolute_error'], l2=0.01):
    regularizer = keras.regularizers.l2(l2)
    feature_layer = keras.layers.DenseFeatures(feature_columns)
    feature_layer_outputs = feature_layer(feature_layer_inputs)
    norm = keras.layers.BatchNormalization()(feature_layer_outputs)
    outputs = keras.layers.Dense(
		1,
		kernel_initializer='normal',
		kernel_regularizer=regularizer,
		activation='linear'
	)(norm)
    
    model = keras.Model(inputs=[v for v in feature_layer_inputs.values()], outputs=outputs)
    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
    return model

In [60]:
# Defining the columns
categorical_cols = ['CHAS', 'RAD']
numeric_cols = ['CRIM', 'ZN', 'INDUS',  'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'B', 'LSTAT']
feature_columns, feature_layer_inputs = define_feature_columns_layers(data, categorical_cols, numeric_cols)
interactions_columns = create_interactions([['RM', 'LSTAT']])

feature_columns += interactions_columns

optimizer = keras.optimizers.Ftrl(learning_rate=0.02)

# Create model with L2 (ridge regression) and L2 norm
model = create_ridge_linreg(
    feature_columns, 
    feature_layer_inputs, 
    optimizer, 
    loss='mean_squared_error', 
    metrics=['mean_absolute_error', 'mean_squared_error'],
    l2=0.01
)
estimator = canned_keras(model, output_dir='logs/Ridge')
train_input_fn = make_input_fn(train, y_train, num_epochs=1400)
test_input_fn = make_input_fn(test, y_test, num_epochs=1, shuffle=False)
estimator.train(train_input_fn)
result = estimator.evaluate(test_input_fn)
print(result)

INFO:tensorflow:Using the Keras model provided.
INFO:tensorflow:Using config: {'_model_dir': 'logs/Ridge', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
<RepeatDataset shapes: ({CRIM: (None,), ZN: (None,), INDUS: (None,), CHAS: (None,), NOX: (None,), RM: (None,), AG

In [66]:
# Lasso Regularization
def create_lasso_linreg(feature_columns, feature_layer_inputs, optimizer, loss='mean_squared_error', metrics=['mean_absolute_error'], l1=0.001):
    regularizer = keras.regularizers.l1(l1)
    feature_layer = keras.layers.DenseFeatures(feature_columns)
    feature_layer_outputs = feature_layer(feature_layer_inputs)
    norm = keras.layers.BatchNormalization()(feature_layer_outputs)
    outputs = keras.layers.Dense(
		1,
		kernel_initializer='normal',
		kernel_regularizer=regularizer,
		activation='linear'
	)(norm)
    
    model = keras.Model(inputs=[v for v in feature_layer_inputs.values()], outputs=outputs)
    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
    return model

In [67]:
# Defining the columns
categorical_cols = ['CHAS', 'RAD']
numeric_cols = ['CRIM', 'ZN', 'INDUS',  'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'B', 'LSTAT']
feature_columns, feature_layer_inputs = define_feature_columns_layers(data, categorical_cols, numeric_cols)
interactions_columns = create_interactions([['RM', 'LSTAT']])

feature_columns += interactions_columns

optimizer = keras.optimizers.Ftrl(learning_rate=0.02)

# Create model with L1 (lasso) and L2 norm
model = create_lasso_linreg(
    feature_columns, 
    feature_layer_inputs, 
    optimizer, 
    loss='mean_squared_error', 
    metrics=['mean_absolute_error', 'mean_squared_error'],
    l1=0.001
)
estimator = canned_keras(model, output_dir='logs/Lasso')
train_input_fn = make_input_fn(train, y_train, num_epochs=1400)
test_input_fn = make_input_fn(test, y_test, num_epochs=1, shuffle=False)
estimator.train(train_input_fn)
result = estimator.evaluate(test_input_fn)
print(result)

INFO:tensorflow:Using the Keras model provided.
INFO:tensorflow:Using config: {'_model_dir': 'logs/Lasso', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
<RepeatDataset shapes: ({CRIM: (None,), ZN: (None,), INDUS: (None,), CHAS: (None,), NOX: (None,), RM: (None,), AG

__Result__

Test results favour Lasso. Could be explained by a noisy variable that had to be exlcuded in order for the model to improve.

By comparing how the Lasso and Ridge Regression algorithms are structured, Lasso will force coefficients to zero, while Ridge Regression will only decrease coefficients close to zero (but never zero), therefore some less useful features will still have an effect on the model causing the results to be slightly different between the 2 regularization methods.

## Extention - Elastic net regression

Elastic net regression is a type of regularization that combines both Lasso and Ridge Regression by adding L1 and L2 regularization terms to the loss function.

$$
L_{elasticNet}(\hat{\beta}) = \sum_{i=1}^{n}(y_i-x'_i\hat{\beta})^2+r\lambda\sum_{j=1}^{m}|\hat{\beta}_j| + \frac{1-r}{2}\lambda \sum_{j=1}^{m}w_j\hat{\beta}_j^2
$$

In [68]:
# ElasticNet Regularization
def create_elasticnet_linreg(feature_columns, feature_layer_inputs, optimizer, loss='mean_squared_error', metrics=['mean_absolute_error'], l1=0.001, l2=0.01):
    regularizer = keras.regularizers.l1_l2(l1, l2)
    feature_layer = keras.layers.DenseFeatures(feature_columns)
    feature_layer_outputs = feature_layer(feature_layer_inputs)
    norm = keras.layers.BatchNormalization()(feature_layer_outputs)
    outputs = keras.layers.Dense(
		1,
		kernel_initializer='normal',
		kernel_regularizer=regularizer,
		activation='linear'
	)(norm)
    
    model = keras.Model(inputs=[v for v in feature_layer_inputs.values()], outputs=outputs)
    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
    return model

In [69]:
# Defining the columns
categorical_cols = ['CHAS', 'RAD']
numeric_cols = ['CRIM', 'ZN', 'INDUS',  'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'B', 'LSTAT']
feature_columns, feature_layer_inputs = define_feature_columns_layers(data, categorical_cols, numeric_cols)
interactions_columns = create_interactions([['RM', 'LSTAT']])

feature_columns += interactions_columns

optimizer = keras.optimizers.Ftrl(learning_rate=0.02)

# Create model with ElasticNet (l1 + l2 regularization) with L2 norm
model = create_elasticnet_linreg(
    feature_columns, 
    feature_layer_inputs, 
    optimizer, 
    loss='mean_squared_error', 
    metrics=['mean_absolute_error', 'mean_squared_error'],
    l1=0.001,
    l2=0.01
)
estimator = canned_keras(model, output_dir='logs/ElasticNet')
train_input_fn = make_input_fn(train, y_train, num_epochs=1400)
test_input_fn = make_input_fn(test, y_test, num_epochs=1, shuffle=False)
estimator.train(train_input_fn)
result = estimator.evaluate(test_input_fn)
print(result)

INFO:tensorflow:Using the Keras model provided.
INFO:tensorflow:Using config: {'_model_dir': 'logs/ElasticNet', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
<RepeatDataset shapes: ({CRIM: (None,), ZN: (None,), INDUS: (None,), CHAS: (None,), NOX: (None,), RM: (None,

__Result__

Model doesn't outperform Lasso, since this problem involves redundant variable in the prediction, therefore, Lasso is still the better choice

# 4e. Implementing Logisitic Regression
Turn linear regression in to a binary classification problem. By applying the sigmoid function to the linear output, it scales the result from zero to one, representing the probability that the data point exists in one class or not. 

## Selecting the Cutoff
By then specified a cutoff of __0.5__, the model can determine which class the outcome belongs to. This value can however be tuned according to the obejctive of the model. 

E.g By increasing the threshold, we reduce the number of class 1 predictions, meaning that the number of false negatives increase. However this would mean that those thare are predicted to be in class 1 will have high likelihood to be in class 1, which those more in between will be classified as 0. This means the model is highly precise at the cost of missing out some potential positive cases.

On the flipside, if the use case for the model was to be used to detect cancer having a higher false positive rate compared to a false negative rate is always better, therefore the threshold of determining a class 1 case might be lower.

A 0.5 threshold means that the model is balanced between false positives and false negativse. The final threshold is always somethign that has to be considered depending on the model's real-world applications

## About the Dataset
Logistic regression to preduict the probability of breast cancer using the Breast Cancer Wisconsin dataset. Predicting the diagnosis from features  that are computed from a digitized image of a __fine needle aspiration (FNA)__.

Dataset can immediately be used as a classification model without further transformations

In [39]:
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import pandas as pd
import tensorflow_datasets as tfds
tfds.disable_progress_bar()

# Data preparation
breast_cancer = 'https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data'
path = tf.keras.utils.get_file(breast_cancer.split('/')[-1], breast_cancer)

columns = ['sample_code', 'clump_thickness', 'cell_size_uniformity', 'cell_shape_uniformity',
           'marginal_adhesion', 'single_epithelial_cell_size', 'bare_nuclei', 'bland_chromatin',
           'normal_nucleoli', 'mitoses', 'class']

data = pd.read_csv(path, header=None, names=columns, na_values=[np.nan, '?'])
# Code to see which rows has null values
# data[data.isnull().any(axis=1)]
data = data.fillna(data.median()) # fill the NA values with the median value of the column

# Splitting the data
np.random.seed(1)
train = data.sample(frac=0.8).copy()
y_train = (train['class'] == 4).astype(int)
train.drop(['sample_code', 'class'], axis=1, inplace=True)

test = data.loc[~data.index.isin(train.index)].copy()
y_test = (test['class'] == 4).astype(int)
test.drop(['sample_code', 'class'], axis=1, inplace=True)

print(f"Distribution of samples: Training samples - {train.shape[0]} | Testing samples - {test.shape[0]}")


Distribution of samples: Training samples - 559 | Testing samples - 140


In [33]:
# Model deifnition - Change the activation function in the single output neuron from 'linear' to 'sigmoid'
def create_logreg(feature_columns, feature_layer_inputs, optimizer, loss='binary_crossentropy', metrics=['accuracy'], l2=0.01):
    regularizer = keras.regularizers.l2(l2)

    feature_layer = keras.layers.DenseFeatures(feature_columns)
    feature_layer_outputs = feature_layer(feature_layer_inputs)
    norm = keras.layers.BatchNormalization()(feature_layer_outputs)
    outputs = keras.layers.Dense(1, 
                                 kernel_initializer='normal', 
                                 kernel_regularizer = regularizer, 
                                 activation='sigmoid')(norm)
    
    model = keras.Model(inputs=[v for v in feature_layer_inputs.values()], outputs=outputs)
    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
    return model

In [34]:
# Run the model
categorical_cols = []
numeric_cols = ['clump_thickness', 'cell_size_uniformity', 'cell_shape_uniformity',
                'marginal_adhesion', 'single_epithelial_cell_size', 'bare_nuclei', 'bland_chromatin',
                'normal_nucleoli', 'mitoses']

feature_columns, feature_layer_inputs = define_feature_columns_layers(data, categorical_cols, numeric_cols)
optimizer = keras.optimizers.Ftrl(learning_rate=0.007)
model = create_logreg(feature_columns, feature_layer_inputs, optimizer, l2=0.01)
estimator = canned_keras(model, output_dir='./logs/logreg')

train_input_fn = make_input_fn(train, y_train, num_epochs=300, batch_size=32)
test_input_fn = make_input_fn(test, y_test, num_epochs=1, shuffle=False)
estimator.train(train_input_fn)
result = estimator.evaluate(test_input_fn)
print(result)

INFO:tensorflow:Using the Keras model provided.
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
INFO:tensorflow:Using config: {'_model_dir': './logs/logreg', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replic

In [35]:
%load_ext tensorboard
%tensorboard --logdir ./logs/logreg

__Result__

Able to achieve good results (95% Accuracy) in spite of a slightly unbalanced target class.

__Summary__
* Logistic regression predictions are based on the sigmoid curve
* When predicting for a multi-class or multi-label, don't have to use different kinds of One Versus All strategies, just increase the number of nodes to match the number of classes that you have
	- Will use a softmax activation instead

# 4f. Using Non-linear Solutions

More complex models that can potentially model data better. These do not subscribe to 1 coefficient to 1 variable.

__Support Vector Machines (SVM)__

Viable option in terms of random features for large-scale kernel machines. Below leverages Keras to obtain a non-linear solution to a classification problem. The non-linear function applied to the features will be a __Random Fourier Features__ function which approximates kernel-based classifiers and regressors.

In [40]:
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import pandas as pd
import tensorflow_datasets as tfds
tfds.disable_progress_bar()

# Importing Random Fourier Features function 
try:
    from tensorflow.python.keras.layers.kernelized import RandomFourierFeatures
except:
    from tensorflow.keras.layers.experimental import RandomFourierFeatures

In [43]:
# Define the model - With additional RandomFourierFeatures function for non-linearity
def create_svc(feature_columns, feature_layer_inputs, optimizer, loss='hinge', metrics=['accuracy'], l2=0.01, output_dim=64, scale=None):
	regularizer = keras.regularizers.l2(l2)
 
	feature_layer = keras.layers.DenseFeatures(feature_columns)
	feature_layer_outputs = feature_layer(feature_layer_inputs)
	norm = keras.layers.BatchNormalization()(feature_layer_outputs)
	rff = RandomFourierFeatures(output_dim=output_dim, scale=scale, kernel_initializer='gaussian')(norm)
	outputs = keras.layers.Dense(
		1,
		kernel_initializer='normal',
		kernel_regularizer=regularizer,
		activation='sigmoid'
	)(rff)
 
	model = keras.Model(inputs=[v for v in feature_layer_inputs.values()], outputs=outputs)
	model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
	return model

In [44]:
# Train the model
categorical_cols = []
numeric_cols = ['clump_thickness', 'cell_size_uniformity', 'cell_shape_uniformity',
                'marginal_adhesion', 'single_epithelial_cell_size', 'bare_nuclei', 'bland_chromatin',
                'normal_nucleoli', 'mitoses']

feature_columns, feature_layer_inputs = define_feature_columns_layers(data, categorical_cols, numeric_cols)
optimizer = keras.optimizers.Adam(learning_rate=0.00005)
model = create_svc(feature_columns, feature_layer_inputs, optimizer, loss='hinge', l2=0.001, output_dim=512)
estimator = canned_keras(model, output_dir='./logs/svc')

train_input_fn = make_input_fn(train, y_train, num_epochs=500, batch_size=512)
test_input_fn = make_input_fn(test, y_test, num_epochs=1, shuffle=False)
estimator.train(train_input_fn)
result = estimator.evaluate(test_input_fn)
print(result)

INFO:tensorflow:Using the Keras model provided.
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
INFO:tensorflow:Using config: {'_model_dir': './logs/svc', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas'

In [46]:

%load_ext tensorboard
%tensorboard --logdir ./logs/svc
%tensorboard dev upload --logdir './logs/svc'

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 5268), started 0:01:27 ago. (Use '!kill 5268' to kill it.)

ERROR: Timed out waiting for TensorBoard to start. It may still be running as pid 15056.

__Result__

Plot is relatively smooth compared ones because of a larger batch. Larger models with more neurons generally perform better with larger batches

__Summary__

* Random Fourier Features helps to mimic the function of SVM kernels (RBF, polynomial, etc.) 
	- able to achieve a lower computational complexity making such an approach feasible for a neural network implementation
* Different non-linear models can be achieved with different loss functions:
	- __Hinge loss__: Sets an SVM model
	- __Logistics loss__: kernel logistic regression (returns class probabilities)
	- __Mean square error__: kernel regression 

# 4g. Wide & Deep Models

_Linear models_ are efficient and interpretable because of __memorization__. Linear models record the association between features (or a group of features) into a single coefficient.

_Neural Networks_ are __generalized__ because their complexity is able to approximate general rules that govern the outcome of a process.

__Wide & Deep models__ blend memorization and generalization because they combine a linear model applied to numeric features together with generalization applied to sparse features, such as categories encoded into a sparse matrix. (https://arxiv.org/pdf/1606.07792.pdf)

The model was originally used as a basis for recommendation systems fro the Google Play Store because each "part" of the model handled different types of data. 

* Wide (regression): features relative to user's characteristics (dense numeric features, binary indicators, or combination of interaction features) which are quite consistent
* Deep (neural network): feature strings representing previous software downloads (spares inputs on large matrices), which are more variable over time

Use the Adult datasent, or Census dataset to predict if income exceeds $50K/annum based on the census data. Available features are quite varied, from age to occupation. Feed the different features to the right side of the Wide & Deep model. `tf.estimator.DNNLinearCombinedEstimator`

In [1]:
import tensorflow as tf
import pandas as pd

In [52]:
# Download the dataset and split into train and test sets
census_dir = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/'
train_path = tf.keras.utils.get_file('adult.data', census_dir + 'adult.data')
test_path = tf.keras.utils.get_file('adult.test', census_dir + 'adult.test')

columns = ['age', 'workclass', 'fnlwgt', 'education', 'education_num',
           'marital_status', 'occupation', 'relationship', 'race', 'gender',
           'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
           'income_bracket']

train_data = pd.read_csv(train_path, header=None, names=columns)
test_data = pd.read_csv(test_path, header=None, names=columns, skiprows=1)
train_data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,gender,capital_gain,capital_loss,hours_per_week,native_country,income_bracket
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [53]:
train_data.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [54]:
# Select a subset of features to be used for model
predictors = ['age', 'workclass', 'education', 'education_num', 'marital_status', 
			  'occupation', 'relationship', 'gender']

# Change the target variable to integer
y_train = (train_data['income_bracket'] == ' >50K').astype(int)
y_test = (test_data['income_bracket'] == ' >50K.').astype(int)

train_data = train_data[predictors]
test_data = test_data[predictors]

# Replace missing values with a mean value
train_data[['age', 'education_num']] = train_data[['age', 'education_num']].fillna(train_data[['age', 'education_num']].mean())
test_data[['age', 'education_num']] = test_data[['age', 'education_num']].fillna(train_data[['age', 'education_num']].mean())

In [58]:
# Functions that deal with the interaction of categorical and numeric columns
def define_feature_columns(data_df, numeric_cols, categorical_cols, embedded_cols, dims=30):
	num_cols = []
	cat_cols = []
	embeddings = []
	
	for feature_name in numeric_cols:
		num_cols.append(tf.feature_column.numeric_column(feature_name, dtype=tf.float32))
	for feature_name in categorical_cols:
		vocabulary = data_df[feature_name].unique()
		cat_cols.append(tf.feature_column.categorical_column_with_vocabulary_list(feature_name, vocabulary))
	for feature_name in embedded_cols:
		vocabulary = data_df[feature_name].unique()
		to_categorical = tf.feature_column.categorical_column_with_vocabulary_list(feature_name, vocabulary)
		embeddings.append(tf.feature_column.embedding_column(to_categorical, dimension=dims))
	
	return num_cols, cat_cols, embeddings
	
# Function to create interaction terms between variables of all types
def create_interactions(interactions_list, buckets=10):
    feature_columns = []
    
    for (a, b) in interactions_list:
        crossed_feature = tf.feature_column.crossed_column([a, b], hash_bucket_size=buckets)
        crossed_feature_one_hot = tf.feature_column.indicator_column(crossed_feature)
        feature_columns.append(crossed_feature_one_hot)

    return feature_columns

In [59]:
# Map different columns and add meaninful interactions - to create final list of features
numeric_cols, categorical_cols, embedding_cols = define_feature_columns(train_data,
                                                                       ['age', 'education_num'],
                                                                       ['gender'],
                                                                       ['workclass', 'education', 'marital_status', 'occupation', 'relationship'],
                                                                       dims=32)
                                                                       # mapping high-dimensional categorical features into fixed lower dimensional numeric space of 32
interactions = create_interactions([['education', 'occupation']], buckets=10)                                    

In [61]:
# Model
estimator = tf.estimator.DNNLinearCombinedClassifier(
	# wide settings
	linear_feature_columns=numeric_cols + categorical_cols + interactions,
	linear_optimizer=tf.keras.optimizers.Ftrl(0.0002),
	# deep settings
	dnn_feature_columns=embedding_cols,
	dnn_hidden_units=[1024, 256, 128, 64],
	dnn_optimizer=tf.keras.optimizers.Adagrad(0.0001),
	model_dir='./logs/wd'
)

INFO:tensorflow:Using default config.
INFO:tensorflow:Using config: {'_model_dir': './logs/wd', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}


In [62]:
def make_input_fn(data_df, label_df, num_epochs=10, shuffle=True, batch_size=256):
    def input_function():
        ds = tf.data.Dataset.from_tensor_slices((dict(data_df), label_df))
        if shuffle:
            ds = ds.shuffle(1000)
        ds = ds.batch(batch_size).repeat(num_epochs)
        return ds
    
    return input_function

In [63]:
# Train estimator
train_input_fn = make_input_fn(train_data, y_train, num_epochs=100, batch_size=256)
test_input_fn = make_input_fn(test_data, y_test, num_epochs=1, shuffle=False)

estimator.train(input_fn=train_input_fn, steps=1500)
results = estimator.evaluate(input_fn=test_input_fn)
print(results)

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
Instructions for updating:
Use Variable.read_value. Variables in 2.X are initialized automatically both in eager and graph (inside tf.defun) contexts.
INFO:tensorflow:Calling model_fn.
Instructions for updating:
Please use `layer.add_weight` method instead.
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Create CheckpointSaverHook.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Calling checkpoint listeners before saving checkpoint 0...
INFO:tensorflow:Saving checkpoints for 0 into ./logs/wd\model.ckpt.
INFO:tensorflow:Calling checkpoint listeners after saving checkpoint 0...
INFO:tensorflow:loss = 0.70521426, step = 0
INFO:tensorflow:global_step/sec: 25.7998
INFO:tensorflow:loss = 0.6665188, step

__Result__

Decent results, but could be better with more tuning

__Summary__

Wide & Depp models represent a way to handle linear models together with more complex approaches involving neural networks. They are usually used for recommendation systems where there are multiple features of different data types, and therefore require different methods to feed into the model.

Estimators are straightfoward and easy to use. They are also efficient, and provide out of the box models to quickly deploy various ML algorithms. Bulk of the work is done by defining an input data function and mapping the features with more suitable functions from `tf.feature_columns`